# YOLO11 Pose v2: train có ô tô và xe máy
Bộ mới giữ nguyên nhóm nguồn: car + green trong train; bv1/cam trong validation. Validation chỉ đánh giá xe máy, chưa đo khả năng tổng quát ô tô độc lập. Chạy từ trên xuống, bật GPU. Không resume checkpoint bộ v1.


In [ ]:
%pip install ultralytics==8.4.183 Pillow==11.3.0 onnx onnxruntime onnxslim tensorboard
import sys, subprocess
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
with open('/content/environment-freeze.txt', 'w') as environment:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, stdout=environment)


## Google Drive và cấu hình chạy
Cho phép Colab kết nối Drive của bạn. Dataset được cache trên Drive; ảnh được giải nén ở ổ local Colab để train nhanh.

- Lần đầu: `RESUME=False`, `RUN_ID=''`. Mặc định train đầy đủ 100 epoch; có thể đặt `FULL_TRAIN=False` để thử 2 epoch.
- Train đầy đủ: sửa `FULL_TRAIN=True`, giữ `RESUME=False`, `RUN_ID=''`, chạy lại từ ô này.
- Sau khi ngắt: chạy lại notebook, đặt đúng `FULL_TRAIN` như lần trước, `RESUME=True`, dán **RUN_ID cũ**. Dataset cache được dùng lại; tiếp tục từ epoch đã backup.
- Không chạy hai phiên Colab đồng thời vào cùng RUN_ID. Khi tạo run mới, thư mục Drive đã tồn tại sẽ bị từ chối để tránh ghi đè.

Mỗi epoch hoàn tất được backup sang hai slot luân phiên, kiểm tra SHA-256 rồi mới cập nhật checkpoint.json. Slot trước vẫn còn nếu quá trình copy bị ngắt. Batch/epoch đang chạy dở có thể phải chạy lại; trước checkpoint đầu tiên chưa có tiến độ có thể phục hồi.

Bộ v2 dùng thư mục Drive/cache riêng. Giữ RUN_ID mới; không dùng RUN_ID/checkpoint v1.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import uuid, re
drive.mount('/content/drive')

FULL_TRAIN = True
RESUME = False
RUN_ID = ''  # Khi resume: dán chính xác RUN_ID được in trong lần chạy trước.
IMGSZ = 960
BATCH = 4
EPOCHS = 100 if FULL_TRAIN else 2
DRIVE_ROOT = Path('/content/drive/MyDrive/nATime-YOLO11-Pose-v2-sourceholdout')
DRIVE_DATASET = DRIVE_ROOT / 'datasets/dataset-mixed-v2-training.zip'
DRIVE_DATASET.parent.mkdir(parents=True, exist_ok=True)
if RESUME:
    assert RUN_ID, 'Cần RUN_ID cũ để tiếp tục'
elif not RUN_ID:
    RUN_ID = ('full-' if FULL_TRAIN else 'smoke-') + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-') + uuid.uuid4().hex[:6]
assert re.fullmatch(r'[A-Za-z0-9_-]{1,100}', RUN_ID), 'RUN_ID chỉ dùng chữ, số, gạch ngang/gạch dưới'
DRIVE_RUN = DRIVE_ROOT / 'runs' / RUN_ID
LOCAL_RUN = Path('/content/natime-pose-runs') / RUN_ID
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
print('RUN_ID:', RUN_ID)
print('Drive:', DRIVE_RUN)
print('Mode:', 'FULL' if FULL_TRAIN else 'SMOKE', '| Resume:', RESUME)


## 1. Nạp gói đã duyệt và xác minh
Đầu ra mới cho mỗi lần chạy, không tái sử dụng thư mục cũ. Dừng nếu hash tool không khớp; cập nhật notebook từ cùng source khi nâng tool.

In [ ]:
from google.colab import files
import zipfile, hashlib, importlib.util, json, tempfile, shutil, os
if DRIVE_DATASET.is_file():
    local_archive = Path(tempfile.mkdtemp(prefix='pose-zip-', dir='/content')) / 'dataset-mixed-v2-training.zip'
    shutil.copyfile(DRIVE_DATASET, local_archive)
    archives = [str(local_archive)]
    print('Dùng dataset đã cache trên Drive')
else:
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('-training.zip')]
    assert len(archives) == 1, 'Chọn đúng một ZIP đã duyệt *-training.zip'
with open(archives[0], 'rb') as package_stream:
    assert hashlib.file_digest(package_stream, 'sha256').hexdigest() == '4f85df9391b55600097b3c1c8f36fbd807d8745507733985b926e31fa9261b3e', 'Sai ZIP v2; chọn dataset-mixed-v2-training.zip'
ROOT = Path(tempfile.mkdtemp(prefix='plate-pose-', dir='/content'))
with zipfile.ZipFile(archives[0]) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)), 'ZIP có tên file trùng'
    assert sum(i.file_size for i in archive.infolist()) < 8 * 1024**3, 'ZIP quá lớn'
    for name in names:
        p = Path(name)
        assert not p.is_absolute() and '..' not in p.parts and chr(92) not in name and ':' not in name
        assert (name in {'images/', 'labels/', 'images/train/', 'images/val/', 'labels/train/', 'labels/val/'} or
                name in {'manifest.json','plate_pose.yaml','prepare_pose_dataset.py'} or
                (p.parts[0] in {'images','labels'} and len(p.parts) == 3 and p.parts[1] in {'train','val'})), name
    tool_bytes = archive.read('prepare_pose_dataset.py')
    assert hashlib.sha256(tool_bytes).hexdigest() == 'ced7280f1122ddae9c5e981d3172bd7d7dc41f0d23b9ad5dbcb9f7b811fa003c', 'Tool không cùng phiên bản notebook'
    archive.extractall(ROOT)
spec = importlib.util.spec_from_file_location('pose_prepare', ROOT/'prepare_pose_dataset.py')
prepare = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prepare)
manifest = prepare.verify(ROOT)
assert hashlib.sha256((ROOT/'manifest.json').read_bytes()).hexdigest() == '05660b261f0c6637bd8d075832c7041ecd7fcea1ac092fc8b9111214699f7712', 'Sai manifest v2'
assert manifest['dataset_version'] == 'mixed-v2-source-holdout'
from collections import Counter
print('Số ảnh theo split/nguồn:', Counter((r['split'], r['group']) for r in manifest['records']))
print('Lưu ý: validation chỉ có xe máy; chưa có test ô tô độc lập.')
print({key: manifest[key] for key in ('original_images','removed_duplicates','excluded_images','actual_val_fraction')})
assert manifest['reviewed'] and manifest['corner_order'] == ['TL','TR','BR','BL']
DATA = ROOT/'colab_pose.yaml'
DATA.write_text(f'path: {ROOT.as_posix()}\ntrain: images/train\nval: images/val\nkpt_shape: [4, 3]\nflip_idx: []\nnames:\n  0: license_plate\n', encoding='utf-8')

if not DRIVE_DATASET.exists():
    temporary = DRIVE_DATASET.with_suffix('.zip.tmp')
    shutil.copyfile(archives[0], temporary)
    with open(archives[0], 'rb') as stream:
        original_hash = hashlib.file_digest(stream, 'sha256').hexdigest()
    with temporary.open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == original_hash
    os.replace(temporary, DRIVE_DATASET)
    print('Dataset đã được lưu vào Drive')


## 2. Kiểm tra overlay
Đây chỉ là mẫu xem lại, không thay thế duyệt toàn bộ góc và nhóm tại máy local. Kiểm tra cả ảnh nhiều biển/biển nhỏ và tập validation.

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display
import random
rng = random.Random(42)
for split in ('train','val'):
    rows = [r for r in manifest['records'] if r['split'] == split]
    for row in rng.sample(rows, min(6, len(rows))):
        image = Image.open(ROOT/row['image']).convert('RGB')
        draw = ImageDraw.Draw(image)
        pts = [(x*image.width, y*image.height) for x,y in row['points']]
        draw.line(pts+[pts[0]], fill='red', width=3)
        for label, point in zip(manifest['corner_order'],pts):
            draw.text(point,label,fill='red',stroke_width=1)
        image.thumbnail((960,540))
        print(split, row['group'], row['id'])
        display(image)


## Checkpoint và bảng hoạt động
Mã hỗ trợ đã nhúng trực tiếp vào ô train; không cần chạy ô helper riêng. Checkpoint vẫn lưu Drive mỗi epoch.


## Theo dõi trực tiếp
Chạy ô TensorBoard **trước** ô train, rồi cuộn lại bảng này trong khi training. Xem Scalars: train/pose_loss, val/pose_loss và metrics/mAP50-95(P) (tên có thể khác theo phiên bản). Log Ultralytics trong ô train vẫn hiển thị epoch, batch, GPU memory và thời gian.

Bảng trạng thái ở ô train cập nhật khoảng 20 giây/lần và tại các mốc epoch/validation/checkpoint. `checkpoint_epoch` là epoch đã lưu lên Drive, khác với epoch đang tính. `status.json` chỉ là dấu hoạt động gần nhất; file cũ không chứng minh runtime vẫn đang chạy.


In [ ]:
from ultralytics import settings
settings.update({'tensorboard': True})
get_ipython().run_line_magic('load_ext', 'tensorboard')
get_ipython().run_line_magic('tensorboard', '--logdir /content/natime-pose-runs')


## Train đầy đủ
Mặc định 100 epoch, ảnh 960, batch 4, early stopping 20 epoch. Bắt đầu từ pretrained YOLO11n Pose; bộ dữ liệu đã đổi nên không resume v1. TensorBoard và checkpoint Drive vẫn hoạt động như trước.


In [ ]:
"""Self-contained code embedded in the Colab notebook; stdlib-only storage core."""
import hashlib
import html
import json
import math
import os
from pathlib import Path
import shutil
import sys
import time
from datetime import datetime, timezone


def file_sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, data):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(data, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def atomic_copy(source, destination):
    source, destination = Path(source), Path(destination)
    temporary = destination.with_name(destination.name + ".tmp")
    with source.open("rb") as src, temporary.open("wb") as dst:
        shutil.copyfileobj(src, dst)
        dst.flush()
        os.fsync(dst.fileno())
    checksum = file_sha256(source)
    if file_sha256(temporary) != checksum:
        raise OSError("Checkpoint copy hash mismatch; previous checkpoint is retained")
    os.replace(temporary, destination)
    return {"file": destination.name, "sha256": checksum, "size": destination.stat().st_size}


class DriveCheckpointStore:
    def __init__(self, run_dir, contract, resume=False):
        self.root = Path(run_dir)
        self.contract = contract
        self.marker = self.root / "checkpoint.json"
        if resume:
            saved = json.loads((self.root / "run-contract.json").read_text(encoding="utf-8"))
            if saved != contract:
                raise ValueError("Run configuration/dataset differs; choose the original configuration or a new RUN_ID")
        else:
            self.root.mkdir(parents=True, exist_ok=False)
            atomic_json(self.root / "run-contract.json", contract)

    def state(self):
        return json.loads(self.marker.read_text(encoding="utf-8")) if self.marker.exists() else None

    def checked_path(self, entry):
        if Path(entry["file"]).name != entry["file"]:
            raise ValueError("Invalid checkpoint path")
        path = self.root / entry["file"]
        if not path.is_file() or path.stat().st_size != entry["size"] or file_sha256(path) != entry["sha256"]:
            raise ValueError("Checkpoint is missing or corrupted")
        return path

    def restore(self, local_dir, allow_completed=False):
        state = self.state()
        if not state:
            raise ValueError("No committed checkpoint; epoch 1 has not been backed up yet")
        if state.get("completed") and not allow_completed:
            raise ValueError("Run is already completed; export best.pt or start a new RUN_ID")
        # If the latest slot is damaged, retain the option to resume the previous epoch.
        for snapshot in (state, state.get("previous")):
            if not snapshot:
                continue
            try:
                last = self.checked_path(snapshot["last"])
                best = self.checked_path(snapshot["best"]) if snapshot.get("best") else None
            except (ValueError, OSError):
                continue
            local = Path(local_dir)
            (local / "weights").mkdir(parents=True, exist_ok=True)
            shutil.copyfile(last, local / "resume.pt")
            if best:
                shutil.copyfile(best, local / "weights/best.pt")
            if (self.root / "logs").exists():
                shutil.copytree(self.root / "logs", local, dirs_exist_ok=True)
            if snapshot is not state:
                atomic_json(self.marker, snapshot)
            return local / "resume.pt", snapshot
        raise ValueError("Both checkpoint generations failed verification; do not start a fresh run over them")

    def backup(self, last, best, epoch, total_epochs):
        previous = self.state()
        slot = "b" if previous and previous["last"]["file"] == "last-a.pt" else "a"
        last_entry = atomic_copy(last, self.root / f"last-{slot}.pt")
        best_entry = atomic_copy(best, self.root / f"best-{slot}.pt") if Path(best).is_file() else None
        snapshot = dict(epoch=epoch, total_epochs=total_epochs, last=last_entry, best=best_entry,
                        updated_at_utc=datetime.now(timezone.utc).isoformat(), completed=False)
        if previous:
            snapshot["previous"] = {key: value for key, value in previous.items() if key != "previous"}
        # The marker changes only after both complete copies have been verified.
        atomic_json(self.marker, snapshot)
        return snapshot

    def sync_logs(self, local_dir):
        # TensorBoard writes asynchronously. Flush its pinned integration before
        # hashing/copying event files, so a growing file cannot look corrupted.
        integration = sys.modules.get("ultralytics.utils.callbacks.tensorboard")
        writer = getattr(integration, "WRITER", None)
        if writer is not None:
            writer.flush()
        local = Path(local_dir)
        logs = self.root / "logs"
        logs.mkdir(exist_ok=True)
        for file in local.iterdir():
            if file.is_file() and (file.name in {"results.csv", "args.yaml", "environment-freeze.txt", "dataset-manifest.json"}
                                   or file.suffix == ".png" or file.name.startswith("events.out.tfevents")):
                atomic_copy(file, logs / file.name)

    def finish(self):
        snapshot = self.state()
        if not snapshot:
            raise ValueError("No checkpoint committed")
        snapshot["completed"] = True
        atomic_json(self.marker, snapshot)


def finite_metrics(values):
    result = {}
    for key, value in values.items():
        try:
            number = float(value)
            if math.isfinite(number):
                result[str(key)] = round(number, 6)
        except (TypeError, ValueError):
            continue
    return result


class TrainingMonitor:
    def __init__(self, store, display_update=None, interval=20):
        self.store = store
        self.display_update = display_update
        self.interval = interval
        self.last_write = 0
        self.trainer = None
        self.batch = 0
        self.val_batch = 0
        self.epoch_started = time.monotonic()
        self.phase = "initializing"

    def update(self, force=False):
        now = time.monotonic()
        if not force and now - self.last_write < self.interval:
            return
        trainer = self.trainer
        losses = getattr(trainer, "tloss", {})
        if not hasattr(losses, "items"):
            losses = dict(zip(getattr(trainer, "loss_names", ()), losses if losses is not None else ()))
        checkpoint = self.store.state()
        status = dict(phase=self.phase, updated_at_utc=datetime.now(timezone.utc).isoformat(),
                      epoch=int(getattr(trainer, "epoch", 0)) + 1,
                      total_epochs=int(getattr(trainer, "epochs", 0)),
                      batch=self.batch, train_batches=len(trainer.train_loader) if trainer else 0,
                      validation_batch=self.val_batch, epoch_seconds=round(now - self.epoch_started, 1),
                      checkpoint_epoch=checkpoint["epoch"] if checkpoint else 0,
                      losses=finite_metrics(losses), metrics=finite_metrics(getattr(trainer, "metrics", {})))
        atomic_json(self.store.root / "status.json", status)
        if self.display_update:
            summary = html.escape(json.dumps({**status["losses"], **status["metrics"]}, ensure_ascii=False))
            self.display_update(
                f"<b>{html.escape(self.phase)}</b> | Epoch {status['epoch']}/{status['total_epochs']} | "
                f"Batch {self.batch}/{status['train_batches']} | Validation batch {self.val_batch} | "
                f"{status['epoch_seconds']} s/epoch | Checkpoint Drive: epoch {status['checkpoint_epoch']}<br>"
                f"<progress value='{self.batch}' max='{max(1, status['train_batches'])}'></progress>"
                f"<pre style='white-space:pre-wrap'>{summary}</pre>"
                f"Cập nhật UTC: {status['updated_at_utc']}")
        self.last_write = now

    def start(self, trainer):
        self.trainer = trainer
        self.phase = "training"
        self.update(True)

    def epoch_start(self, trainer):
        self.trainer = trainer
        self.batch = 0
        self.val_batch = 0
        self.phase = "training"
        self.epoch_started = time.monotonic()
        self.update(True)

    def batch_end(self, trainer):
        self.trainer = trainer
        self.batch += 1
        self.update()

    def val_start(self, _validator):
        self.phase = "validating"
        self.val_batch = 0
        self.update(True)

    def val_batch_end(self, _validator):
        self.val_batch += 1
        self.update()

    def saved(self, trainer):
        self.trainer = trainer
        self.phase = "saving_checkpoint"
        self.update(True)
        self.store.backup(trainer.last, trainer.best, int(trainer.epoch) + 1, int(trainer.epochs))
        self.store.sync_logs(trainer.save_dir)
        self.phase = "epoch_saved"
        self.update(True)

    def finished(self, trainer):
        self.trainer = trainer
        self.store.sync_logs(trainer.save_dir)
        self.store.finish()
        self.phase = "completed"
        self.update(True)


from ultralytics import YOLO, __version__
from IPython.display import display, HTML
import torch, shutil, json, hashlib
assert torch.cuda.is_available(), 'Cần GPU Colab'
assert __version__ == '8.4.183', 'Dùng đúng phiên bản đã pin để resume nhất quán'
prepare.verify(ROOT)
contract = dict(manifest_sha256=file_sha256(ROOT/'manifest.json'),
                full_train=FULL_TRAIN, epochs=EPOCHS, imgsz=IMGSZ,
                kpt_shape=[4,3], ultralytics=__version__)
store = DriveCheckpointStore(DRIVE_RUN, contract, resume=RESUME)
already_finished = False
if RESUME:
    checkpoint, saved = store.restore(LOCAL_RUN, allow_completed=True)
    already_finished = saved.get('completed', False) or saved['epoch'] >= EPOCHS
    model = YOLO(str(checkpoint))
    print('Checkpoint Drive đã xác minh:', saved['epoch'], '/', saved['total_epochs'])
else:
    model = YOLO('yolo11n-pose.pt')

shutil.copyfile('/content/environment-freeze.txt', LOCAL_RUN/'environment-freeze.txt')
shutil.copyfile(ROOT/'manifest.json', LOCAL_RUN/'dataset-manifest.json')
atomic_copy(LOCAL_RUN/'environment-freeze.txt', DRIVE_RUN/'environment-freeze.txt')
atomic_copy(LOCAL_RUN/'dataset-manifest.json', DRIVE_RUN/'dataset-manifest.json')
panel = display(HTML('Đang chuẩn bị training...'), display_id=True)
monitor = TrainingMonitor(store, lambda markup: panel.update(HTML(markup)))
for event, callback in {
    'on_train_start': monitor.start,
    'on_train_epoch_start': monitor.epoch_start,
    'on_train_batch_end': monitor.batch_end,
    'on_val_start': monitor.val_start,
    'on_val_batch_end': monitor.val_batch_end,
    'on_model_save': monitor.saved,
    'on_train_end': monitor.finished,
}.items():
    model.add_callback(event, callback)

if already_finished:
    panel.update(HTML('Lượt train đã hoàn tất. Có thể chạy ô export.'))
    print('Run đã hoàn tất. Có thể chạy ô export; không train lại.')
else:
    try:
        if RESUME:
            # Pinned trainer restores optimizer/scaler/EMA and updates the new dataset YAML path.
            results = model.train(resume=True, data=str(DATA), device=0,
                                  batch=BATCH, workers=2, save_dir=str(LOCAL_RUN))
        else:
            results = model.train(data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                                  device=0, workers=2, seed=42, deterministic=True,
                                  fliplr=0.0, flipud=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0,
                                  degrees=0.0, perspective=0.0, patience=20,
                                  project=str(LOCAL_RUN.parent), name=RUN_ID,
                                  exist_ok=True, save=True, save_period=-1, plots=True)
    except BaseException:
        monitor.phase = 'interrupted'
        try:
            monitor.update(True)
        except Exception:
            print('Không ghi được status lên Drive. Kiểm tra checkpoint.json trước khi resume.')
        raise
RUN = LOCAL_RUN
print('RUN_ID:', RUN_ID, '| Drive checkpoints:', DRIVE_RUN)
print('Đã lưu:', store.state())


## Xem trạng thái đã lưu / sau khi reconnect
Sau khi mount Drive và chọn đúng RUN_ID, có thể chạy ô này để đọc lần hoạt động gần nhất. Khi Colab bị ngắt, chạy lại các ô thiết lập và đặt RESUME=True; TensorBoard sẽ đọc cả log cũ được phục hồi. Nếu run đã hoàn tất thì notebook bỏ qua train và cho export.


In [ ]:
import json
from datetime import datetime, timezone
status_path = DRIVE_RUN/'status.json'
if status_path.exists():
    status = json.loads(status_path.read_text(encoding='utf-8'))
    age = (datetime.now(timezone.utc) - datetime.fromisoformat(status['updated_at_utc'])).total_seconds()
    print(json.dumps(status, ensure_ascii=False, indent=2))
    print(f'Lần cập nhật gần nhất cách đây {age:.0f} giây; đây không phải kiểm tra runtime còn sống.')
else:
    print('Chưa có trạng thái hoạt động cho RUN_ID này.')
marker = DRIVE_RUN/'checkpoint.json'
if marker.exists():
    print('Checkpoint đã commit:', json.loads(marker.read_text(encoding='utf-8')))


## 4. Đánh giá best.pt và export
Xem box/pose metrics và dự đoán validation, kiểm tra sai số góc/ảnh warp trước khi chấp nhận. Không dùng mAP làm bằng chứng OCR đúng 100%. Export chưa có nghĩa tương thích parser Gate; cần kiểm tra riêng trước triển khai.
Validation v2 chỉ có xe máy. Chỉ số này không đại diện chất lượng ô tô; cần test ô tô mới độc lập.


In [ ]:
assert FULL_TRAIN, 'Smoke không được xuất thành model chính thức; chạy lại train đầy đủ khi đã sẵn sàng'
best = RUN/'weights/best.pt'
assert best.is_file()
trained = YOLO(str(best))
assert list(trained.model.model[-1].kpt_shape) == [4,3], 'Model không phải 4 điểm'
metrics = trained.val(data=str(DATA), imgsz=IMGSZ, device=0, plots=True)
print(metrics.results_dict)
onnx_path = trained.export(format='onnx', imgsz=IMGSZ, batch=1, dynamic=False,
                          opset=12, simplify=True, nms=False)
import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(onnx_path))
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
assert session.get_inputs()[0].shape == [1,3,IMGSZ,IMGSZ]
shape = session.get_outputs()[0].shape
assert len(shape) == 3 and shape[0] == 1 and shape[1] == 17, shape
exported = YOLO(str(onnx_path), task='pose')
onnx_metrics = exported.val(data=str(DATA), imgsz=IMGSZ, device='cpu', batch=1, plots=False)
print('PyTorch:', metrics.results_dict, 'ONNX:', onnx_metrics.results_dict)
target = RUN/'yolo11n-plate-pose.onnx'
shutil.copyfile(onnx_path,target)
contract = dict(corner_order=manifest['corner_order'], kpt_shape=[4,3], imgsz=IMGSZ,
                onnx_sha256=hashlib.sha256(target.read_bytes()).hexdigest(),
                onnx_output_shape=shape, pytorch_metrics=metrics.results_dict,
                onnx_metrics=onnx_metrics.results_dict,
                status='requires_corner_warp_and_Gate_compatibility_acceptance')
(RUN/'model-contract.json').write_text(json.dumps(contract,indent=2),encoding='utf-8')
for path in (target, RUN/'model-contract.json', RUN/'environment-freeze.txt'):
    files.download(str(path))

# Keep exports on Drive too; resumable training checkpoints stay unchanged.
exports = DRIVE_RUN/'exports'
exports.mkdir(exist_ok=True)
for artifact in (target, RUN/'model-contract.json', RUN/'environment-freeze.txt', best):
    atomic_copy(artifact, exports/artifact.name)
print('Đã lưu model và báo cáo vào:', exports)
